# B3 · Tuning XGBoost with a random search (and what it is worth)

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 60 minutes · **extended path**

## Objective

Search the XGBoost hyperparameters with `RandomizedSearchCV` (20 combinations over `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `reg_lambda`, `gamma`) in a 3-fold cross-validation, then fix the rounds with **early stopping** on a validation set and evaluate once on the test set.
Compare with the default parameters and measure how much the optimization is worth.

## Dataset

`ds.loans(n=5000)` with missing values (XGBoost handles them), features `ratio, age, income, job_tenure, amount` plus dummies of `contract_type`. Split: 60% training, 20% validation, 20% test. `learning_rate=0.05` fixed.

## Educational purpose

Learn to set up a realistic tuning and to **measure its gain honestly**: random search over XGBoost hyperparameters is effective, but the advantage over the defaults is usually small (1-2 points of AUC), and it has to be compared with the uncertainty of the estimate.

## Task

1. Compute `default_auc` on the test set: `XGBClassifier` with defaults and `learning_rate=0.05`, `n_estimators` chosen with early stopping on the validation set (`early_stopping_rounds=30`).
2. `RandomizedSearchCV` (20 iterations, `cv=StratifiedKFold(3, shuffle=True, random_state=0)`, `scoring="roc_auc"`, `random_state=0`, `n_jobs=1`) on the **training set only** with `n_estimators=200`, `learning_rate=0.05` fixed and the parameter distribution given in `space`. Save `best` (`best_params_`), `cv_auc` (`best_score_`) and `n_fits` (number of trainings: 20 combinations × 3 folds).
3. With the best parameters retrain with early stopping on the validation set and save `tuned_auc` (AUC on the test set) and `final_rounds`.
4. Save `gain = tuned_auc − default_auc` and `within_noise` (True if `gain` is smaller than `2 * 0.01`: the typical uncertainty of an AUC on 1000 examples is about 1 point).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

df = ds.loans(n=5000)
df["ratio"] = df["amount"] / df["income"]
X = pd.concat([df[["ratio", "age", "income", "job_tenure", "amount"]], pd.get_dummies(df["contract_type"], dtype=float)], axis=1)
y = df["defaulted"]
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=0)
space = {"max_depth": randint(2, 7), "min_child_weight": randint(1, 30), "subsample": uniform(0.5, 0.5), "colsample_bytree": uniform(0.5, 0.5),
         "reg_lambda": [0.1, 1, 5, 20], "gamma": [0, 0.5, 2, 5]}
base = dict(learning_rate=0.05, eval_metric="auc", n_jobs=-1, random_state=0)

## Your work

In [ ]:
default_auc = None

In [ ]:
search = None
best = None
cv_auc = None
n_fits = None

In [ ]:
tuned_auc = None
final_rounds = None
gain = None
within_noise = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`search = RandomizedSearchCV(xgb.XGBClassifier(n_estimators=200, **base), space, n_iter=20, cv=StratifiedKFold(3, shuffle=True, random_state=0), scoring="roc_auc", random_state=0, n_jobs=1).fit(X_tr, y_tr)`; `search.best_params_`. Final rounds: `m = xgb.XGBClassifier(n_estimators=2000, early_stopping_rounds=30, **base, **best).fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)`; `m.best_iteration + 1`.'):
    check(0.7 < default_auc < 0.87, "test AUC with the defaults between 0.70 and 0.87", "XGBClassifier(learning_rate=0.05, ...) with early stopping")
    check(set(best) == set(space) and n_fits == 60 and 0.72 < cv_auc < 0.87, "the search returns the six parameters; 60 trainings; cross-validation AUC between 0.72 and 0.87", "RandomizedSearchCV(n_iter=20, cv=3)")
    check(best["max_depth"] <= 4 and best["min_child_weight"] >= 5, "the signal is weak and noisy: the search prefers shallow trees (depth 4 or less) and a high min_child_weight (5 or more)", "conservative hyperparameters")
    check(0.7 < tuned_auc < 0.87 and final_rounds >= 20, "test AUC of the optimized model between 0.70 and 0.87 with at least 20 rounds", "early stopping on the validation set")
    check(close(gain, tuned_auc - default_auc) and abs(gain) < 0.04 and within_noise is True, "the gain over the defaults is small (less than 4 points, within the noise)", "gain = tuned_auc - default_auc")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_tuning_with_random_search.ipynb`. Open it after you pass the check, or after you have really tried.